In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from pymeasure.instruments.agilent import AgilentB1500

In [ ]:
# Utils
def double_sweep(arr):
    """
    주어진 배열의 forward sweep와 reverse sweep을 연결하여 double sweep 생성
    """
    return np.concatenate((arr, arr[::-1]))

def measure_current(b1500, vgs, vds, source_range=11):
    b1500.check_idle()
    
    b1500.smu1.force(source_type='voltage', source_range=source_range, output=vgs)
    b1500.smu2.force(source_type='voltage', source_range=source_range, output=vds)

    time.sleep(0.1)

    b1500.check_errors()
    b1500.clear_buffer()
    b1500.clear_timer()
    b1500.send_trigger()

    b1500.check_idle()

    try:
        response = b1500.smu3.ask("READ?")
        current = float(response)
    except Exception as e:
        print(f"Error reading current: {e}")
        current = 0.0
    
    # sweep constant sources back to 0V
    b1500.smu1.ramp_source('VOLTAGE','Auto Ranging',0,stepsize=0.1,pause=20e-3)
    b1500.smu2.ramp_source('VOLTAGE','Auto Ranging',0,stepsize=0.1,pause=20e-3)
    
    return current


In [ ]:
# Measuremenets
def measure_transfer_curve(b1500):
    """
    전달 곡선 측정 함수.
    
    조건:
      - VGS: -3 V ~ 3 V, 약 100포인트, double sweep
      - VDS: 0.1 V 및 3 V 고정
      
    반환:
      vgs_sweep: 사용된 VGS 스윕 배열
      transfer_data: 각 고정 VDS에 대해 (VGS, ID) 데이터 배열을 딕셔너리로 반환
    """
    num_points = 100  # VGS 스윕 포인트 수
    vgs_forward = np.linspace(-3, 3, num_points)
    vgs_sweep = double_sweep(vgs_forward)
    
    # 고정 드레인 전압 (Drain Voltage)
    drain_biases = [0.1, 3.0]
    transfer_data = {}
    
    for vds in drain_biases:
        data = []
        # 드레인 채널에 고정 전압 설정 (필요한 경우 별도 설정 추가)
        b1500.source2.voltage = vds
        for vgs in vgs_sweep:
            id_measured = measure_current(b1500, vgs, vds)
            data.append((vgs, id_measured))
        transfer_data[vds] = np.array(data)
    
    return vgs_sweep, transfer_data

def measure_output_curve(b1500):
    """
    출력 곡선 측정 함수.
    
    조건:
      - VDS: 0 V ~ 3 V, 약 100포인트, double sweep
      - VGS: 3 V부터 0 V까지 0.3 V step, double sweep
      
    반환:
      vds_sweep: 사용된 VDS 스윕 배열
      output_data: 각 고정 VGS에 대해 (VDS, ID) 데이터 배열을 딕셔너리로 반환
    """
    num_points = 100  # VDS 스윕 포인트 수
    vds_forward = np.linspace(0, 3, num_points)
    vds_sweep = double_sweep(vds_forward)
    
    # VGS 스윕: 3 V에서 0 V로 0.3 V step, double sweep
    vgs_forward = np.arange(3, -0.001, -0.3)  # 0 V 포함
    vgs_sweep = double_sweep(vgs_forward)
    
    output_data = {}
    for vgs in vgs_sweep:
        data = []
        # 게이트 채널에 고정 전압 설정
        b1500.source1.voltage = vgs
        for vds in vds_sweep:
            id_measured = measure_current(b1500, vgs, vds)
            data.append((vds, id_measured))
        output_data[vgs] = np.array(data)
    
    return vds_sweep, output_data

In [ ]:
# Plotting
def plot_transfer_curve(transfer_data):
    """
    전달 곡선 결과 플롯 함수
    """
    plt.figure(figsize=(8, 5))
    for vds, data in transfer_data.items():
        plt.plot(data[:, 0], data[:, 1], marker='.', linestyle='-', label=f"VDS = {vds} V")
    plt.xlabel("VGS (V)")
    plt.ylabel("ID (A)")
    plt.title("Transfer Curve (ID-VGS)")
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_output_curve(output_data):
    """
    출력 곡선 결과 플롯 함수 (예시: 하나의 고정 VGS에 대해 플롯)
    """
    # 예시로, 첫 번째 VGS 값을 선택
    example_vgs = list(output_data.keys())[0]
    data = output_data[example_vgs]
    plt.figure(figsize=(8, 5))
    plt.plot(data[:, 0], data[:, 1], marker='.', linestyle='-')
    plt.xlabel("VDS (V)")
    plt.ylabel("ID (A)")
    plt.title(f"Output Curve (ID-VDS) at VGS = {example_vgs:.2f} V")
    plt.grid(True)
    plt.show()

In [ ]:
# 계측기 연결 (아래 "GPIB::20"은 계측기 주소입니다. 실제 환경에 맞게 수정하세요.)
b1500 = AgilentB1500("GPIB::20")

# 계측기 초기화 및 설정
b1500.reset()  # 계측기 초기화
# 채널 설정: Source1을 게이트 (Gate), Source2를 드레인 (Drain)으로 사용
b1500.source1.compliance = 1e-3  # 예시: 1 mA compliance
b1500.source2.compliance = 1e-3  # 예시: 1 mA compliance

# 필요한 추가 설정 (예: 출력 활성화 등)
# b1500.enable_output()  # 실제 사용 시 계측기의 출력 채널 활성화 명령 추가

# 1. 전달 곡선 (Transfer Curve, ID-VGS) 측정
print("전달 곡선 측정 시작...")
vgs_sweep, transfer_results = measure_transfer_curve(b1500)
print("전달 곡선 측정 완료.")

# 2. 출력 곡선 (Output Curve, ID-VDS) 측정
print("출력 곡선 측정 시작...")
vds_sweep, output_results = measure_output_curve(b1500)
print("출력 곡선 측정 완료.")

# 결과 플롯 (원하는 경우 데이터 저장 또는 추가 분석 수행)
plot_transfer_curve(transfer_results)
plot_output_curve(output_results)

# 계측기 종료 (연결 해제)
b1500.shutdown()
print("계측기 연결 종료.")